In [1]:
import os
import json
import time
from typing import List, Tuple, Dict, Any, Optional, Union
import pytz
from datetime import datetime

import pandas as pd
from tqdm import tqdm
import gspread
from google.oauth2.service_account import Credentials

import cloudvolume as cv
import caveclient

In [5]:
# if first time running this, you will need to be added as a test user (or create service account)
# follow tutorial "For End Users: Using OAuth Client ID": https://docs.gspread.org/en/v6.1.4/oauth2.html
# you will need to add credentials.json to C:\Users\<user>\AppData\Roaming\gspread

# run this the first time you connect
# it should open an auth URL to sign in using your Google credentials
# will add a new authorized_user.json to that same folder C:\Users\username\AppData\Roaming\gspread

# if you haven't connected in some time, you'll need to delete authorized_user.json file located in:
# C:\Users\username\AppData\Roaming\gspread
# if you have connected recently ignore this step, running below should work

gc = gspread.oauth()  # this will guide you through a one-time auth flow


Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=545485027901-vjd7io04c9hutcn76de77lmko4pgbe87.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A60244%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fspreadsheets+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive&state=dUr9ILaINLveIG1M5uCfDXkpcYpmyR&access_type=offline


In [6]:
# test it worked
# https://docs.google.com/spreadsheets/d/1GOvNgj-aswsxdzJRniCobCd384c28gB0rPcXCKcuLEc/edit?gid=367101981#gid=367101981
sheet_id = '1GOvNgj-aswsxdzJRniCobCd384c28gB0rPcXCKcuLEc'
sh = gc.open_by_key(sheet_id)
sh

<Spreadsheet 'update_root_test' id:1GOvNgj-aswsxdzJRniCobCd384c28gB0rPcXCKcuLEc>

In [10]:
SPECIES = "megalopta"
CAVE_SERVER = "https://global.connectomics.braininbrain.org"
DATASTACK_NAME = "megalopta_pb1_datastack"


if DATASTACK_NAME == 'megalopta_fb_eb_datastack':
    GRAPHENE_SEG = "graphene://https://local.cave.braininbrain.org/segmentation/table/megalopta_FB_EB_v3"
elif DATASTACK_NAME == 'megalopta_pb1_datastack':
    GRAPHENE_SEG = "graphene://https://local.cave.braininbrain.org/segmentation/table/megalopta_PB1_v2a"
elif DATASTACK_NAME == 'megalopta_pb2_datastack':
    GRAPHENE_SEG = "graphene://https://local.cave.braininbrain.org/segmentation/table/megalopta_PB2"
elif DATASTACK_NAME == 'eciton_NO_datastack_v5':
    GRAPHENE_SEG = "graphene://https://local.cave.braininbrain.org/segmentation/table/eciton_NO_v5"
elif DATASTACK_NAME == 'eciton_FB_datastack':
    GRAPHENE_SEG = "graphene://https://local.cave.braininbrain.org/segmentation/table/heinze_eciton_FB"
elif DATASTACK_NAME == 'eciton_PB_datastack':
    GRAPHENE_SEG = "graphene://https://local.cave.braininbrain.org/segmentation/table/heinze_eciton_PB_v1"

print("Initializing CAVE client...")
client = caveclient.CAVEclient(server_address=CAVE_SERVER)
client = caveclient.CAVEclient(datastack_name=DATASTACK_NAME)
cg = client.chunkedgraph
print("CAVE client ready.")

print("Initializing CloudVolume (graphene)...")
vol = cv.CloudVolume(GRAPHENE_SEG, use_https=True, progress=False)
print("CloudVolume ready.")

Initializing CAVE client...
CAVE client ready.
Initializing CloudVolume (graphene)...
CloudVolume ready.


In [23]:
if SPECIES == "megalopta":
    if DATASTACK_NAME == 'megalopta_fb_eb_datastack':
        epg = pd.read_csv('../syntables/updated_google_sheets/Megalopta_EB_preprint_FINAL - FBEB_EPG.csv', dtype={'Root ID': str})
        pen = pd.read_csv('../syntables/updated_google_sheets/Megalopta_EB_preprint_FINAL - FBEB_PEN.csv', dtype={'Root ID': str})
        er = pd.read_csv('../syntables/updated_google_sheets/Megalopta_EB_preprint_FINAL - FBEB_ER.csv', dtype={'Root ID': str})
        #er = er[er['Completed']==True]
        nametable = pd.concat((epg, pen, er))
    elif DATASTACK_NAME == 'megalopta_pb1_datastack':
        epg = pd.read_csv('../syntables/updated_google_sheets/Megalopta_PB_preprint_FINAL - PB1_EPG.csv', dtype={'Root ID': str})
        pen = pd.read_csv('../syntables/updated_google_sheets/Megalopta_PB_preprint_FINAL - PB1_PEN.csv', dtype={'Root ID': str}) 
        d7 = pd.read_csv('../syntables/updated_google_sheets/Megalopta_PB_preprint_FINAL - PB1_delta7.csv', dtype={'Root ID': str})
        nametable = pd.concat((epg, pen, d7))
    elif DATASTACK_NAME == 'megalopta_pb2_datastack':
        d7 = pd.read_csv('../syntables/updated_google_sheets/Megalopta_PB2_preprint_FINAL - PB2_delta7.csv', dtype={'Root ID': str})
        epg = pd.read_csv('../syntables/updated_google_sheets/Megalopta_PB2_preprint_FINAL - PB2_EPG.csv', dtype={'Root ID': str})
        nametable = pd.concat((d7, epg))
    elif DATASTACK_NAME == 'megalopta_no_r_datastack': 
        lno = pd.read_csv('../syntables/updated_google_sheets/Megalopta_NOr neuron_CAVE progress - NOr_LNO.csv', dtype={'Root ID': str})
        pen = pd.read_csv('../syntables/updated_google_sheets/Megalopta_NOr neuron_CAVE progress - NOr_PEN.csv', dtype={'Root ID': str})
        nametable = pd.concat((lno, pen))

nametable

,Catmaid name,CATMAID skid,Latest segment IDs,Starting coordinate,Supervoxel ID,Root ID,Root ID updated,Proofreader,Latest update,Ongoing,To split,Completed,Comments,Cell type proposal
0,EPG_L2_55944,55944,576460752521344750,"34780, 7584, 1646",84567562460932612,576460752521344750,2026-03-19,Nina,12/10/2023,False,False,True,partial cell only - but as complete as possible,NaN
1,EPG_L2_55952,55952,576460752510496133,"32249, 7860, 1809",78956204868541358,576460752510496133,2026-03-19,Nina,06/10/2023,False,False,True,"partial cell only, some cube chunks remain; 57...",NaN
2,EPG_L3_54939,54939,"576460752494001689, 576460752504703640","36236, 10336, 1597",88050815297660317,576460752494001689,2026-03-19,Stanley/Marcel/Atticus,06/12/2023,False,True,True,"completed, 2 segments remain due to NoneType e...",NaN
3,EPG_L3_57518,57518,"576460752372935125, 576460752440175329, 576460...","37061, 10733, 1413",89193757634757229,576460752497676789,2026-03-19,Stanley/Atticus,08/09/2024,False,False,True,"completed, 12 segments remain due to NoneType ...",NaN
4,EPG_L3_57529,57529,"576460752460921048, 576460752509771516","36487, 10322, 1612",88050815297667303,576460752460921048,2026-03-19,Stanley/Atticus,04/12/2023,False,False,True,"completed, 2 segments remain due to NoneType e...",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37,delta7_R_L2L10R7_83213,83213,576460752550908713,"45700, 5862, 1397",108158134190621911,576460752550908713,2026-03-19,Nina,24/06/2024,False,False,True,NaN,delta7
38,delta7_R_L2L10R7_83722,83722,576460752608580253,"45608, 5710, 1503",108158409068538182,576460752608580253,2026-03-19,Nina,16/08/2024,False,False,True,"MS updated, this seems to overlap with L3 a lo...",delta7
39,delta7_R_L2L10R7_84178,84178,576460752490172734,"44477, 6529, 1525",105924201440921081,576460752490172734,2026-03-19,Nina/Marcel,16/08/2024,False,False,True,NaN,delta7
40,delta7_R_L2L10R7_84473,84473,"576460752543874747, 576460752545051043","43885, 7444, 1422",104833211028268380,576460752543874747,2026-03-19,Laia/Stanley,NaN,False,False,True,"the two last segments cannot be joined, overla...",delta7


In [25]:
# make backbone coords tuples (these columns should prob be renamed in Google Sheets)
nametable["Starting coordinate"] = nametable["Starting coordinate"].apply(
    lambda s: tuple(map(int, s.split(",")))
)

In [27]:
coords = nametable.loc[
    nametable["Catmaid name"].str.contains("L1L9", na=False),
    "Starting coordinate"
].tolist()

coords

[(37464, 7360, 1329),
 (41943, 7451, 1278),
 (42737, 7140, 1303),
 (43373, 7128, 1526),
 (44285, 6875, 1573),
 (45427, 7550, 1549)]

In [32]:
nametable[nametable["Catmaid name"].str.contains("L1L9", na=False)]

,Catmaid name,CATMAID skid,Latest segment IDs,Starting coordinate,Supervoxel ID,Root ID,Root ID updated,Proofreader,Latest update,Ongoing,To split,Completed,Comments,Cell type proposal
0,delta7_L_L1L9_81004,81004,576460752545395043,"(37464, 7360, 1329)",90196237361395740,576460752545395043,2026-03-19,Nina,21/03/2024,False,False,True,"complete (some dendrites with unclear endings,...",delta7
1,delta7_L_L1L9_84275,84275,576460752525921670,"(41943, 7451, 1278)",100329336522933280,576460752525921670,2026-03-19,Nina,21/08/2024,False,False,True,NaN,delta7
2,delta7_L_L1L9_81014,81014,576460752581659199,"(42737, 7140, 1303)",102563544150600591,576460752581659199,2026-03-19,Nina,15/04/2024,False,False,True,NaN,delta7
3,delta7_L_L1L9_76634,76634,576460752542700737,"(43373, 7128, 1526)",103689993813283301,576460752542700737,2026-03-19,Atticus,16/08/2024,False,False,True,"completed (double checked, much less thick bra...",delta7
4,delta7_L_L1L9_82977,82977,576460752513584969,"(44285, 6875, 1573)",105941793627048306,576460752513584969,2026-03-19,Nina,09/04/2024,False,False,True,NaN,delta7
5,delta7_L_L1L9_78122,78122,576460752485062902,"(45427, 7550, 1549)",108211185626728382,576460752485062902,2026-03-19,Matilda,23/01/2024,False,False,True,Faulty message when I try to cut 5764607525470...,delta7


In [ ]:
namelist = []
coords = []



In [30]:
coord_supervox_dict = vol.scattered_points(coords)

coord_supervox_dict

{(37464, 7360, 1329): 90196237361395740,
 (45427, 7550, 1549): 108211185626728382,
 (42737, 7140, 1303): 102563544150600591,
 (41943, 7451, 1278): 100329336522933280,
 (43373, 7128, 1526): 103689993813283301,
 (44285, 6875, 1573): 105941793627048306}

In [17]:
supervox[coords[0]]

90196237361395740

In [31]:
for coord, supervox_id in coord_supervox_dict.items():
   root = cg.get_roots()
    print(coord)


(37464, 7360, 1329)
(45427, 7550, 1549)
(42737, 7140, 1303)
(41943, 7451, 1278)
(43373, 7128, 1526)
(44285, 6875, 1573)


# need to update to always replace supervox

In [21]:
'''uses neuron backbone coordinate to find supervoxel id,
then uses supervoxel id to update root id'''
 
# -----------------------
# USER CONFIG
# -----------------------

# to do: user options
# spreadsheet_title_list = ['Megalopta_FBEB', 'Megalopta_PB', 'Megalopta_NO', "Eciton_FBEB", "Eciton_PB", "Eciton_NO"]

SPREADSHEET_NAME = 'Megalopta_PB2'
SHEET_NAMES = []  # leave blank to process all tabs ---- otherwise use sheet names 'FBEB_EPG', 'FBEB_PEN', PB1_delta7, PB1_EPG, PB1_PEN

####################################################################
if SPREADSHEET_NAME == 'Megalopta_FBEB': 
    # obtain from URL following /d/<spreadsheet_id>/edit...
    # https://docs.google.com/spreadsheets/d/1kMFNGohw1P6G1HZq-E91GoU2SVtEu3XZpbXHT0Qn7uI/edit?gid=40626915#gid=40626915
    SPREADSHEET_ID = '1kMFNGohw1P6G1HZq-E91GoU2SVtEu3XZpbXHT0Qn7uI'
elif SPREADSHEET_NAME == 'Megalopta_PB':
    SPREADSHEET_ID = '1q-PJWQRdB7zlWqQMl0skg--FcuJftS0J34LVxuPmwns'
elif SPREADSHEET_NAME == 'Megalopta_PB2':
    SPREADSHEET_ID = '1MGn0yT73P7jB9SIw6uJAyr6_baTvAOImNTATJPseK4I'
elif SPREADSHEET_NAME == 'Eciton_NO':
    SPREADSHEET_ID = '1TXPC_S-8pGPFVU_dthR5ixdvojaQax0Kxuv60ViUniw'
elif SPREADSHEET_NAME == 'Eciton_FBEB':
    SPREADSHEET_ID = '1CITAtn02WyC89NdixXNfviMOChshumPoVrnRhPZDW3M'
elif SPREADSHEET_NAME == 'Eciton_PB':
    SPREADSHEET_ID = '12eziKV5Rb6U6tm5oAzm8LXi5WPgsRZPkCV8QMbydbFE'
# https://docs.google.com/spreadsheets/d/12eziKV5Rb6U6tm5oAzm8LXi5WPgsRZPkCV8QMbydbFE/edit?pli=1&gid=0#gid=0

# 2) Column names (must match your Google Sheet headers exactly)
COL_COORD = "Starting coordinate"
COL_SUPER = "Supervoxel ID"
COL_ROOT = "Root ID"
COL_ROOT_UPDATED = "Root ID updated"
RUN_DATE_STR = datetime.now(pytz.timezone("Australia/Sydney")).strftime("%Y-%m-%d")

# 3) CAVE / CloudVolume config
# Example CAVE datastack + Graphene segmentation path (adjust to your env)
CAVE_SERVER = "https://global.connectomics.braininbrain.org"

if SPREADSHEET_NAME == 'Megalopta_FBEB':
    DATASTACK_NAME = "megalopta_fb_eb_datastack"
elif SPREADSHEET_NAME == 'Megalopta_PB':
    DATASTACK_NAME = "megalopta_pb1_datastack"  
elif SPREADSHEET_NAME == 'Megalopta_PB2':
    DATASTACK_NAME = "megalopta_pb2_datastack"
elif SPREADSHEET_NAME == 'Eciton_NO':
    DATASTACK_NAME = "eciton_NO_datastack_v5"
elif SPREADSHEET_NAME == 'Eciton_FBEB':
    DATASTACK_NAME = "eciton_FB_datastack"
elif SPREADSHEET_NAME == 'Eciton_PB':
    DATASTACK_NAME = "eciton_PB_datastack"

if SPREADSHEET_NAME == 'Megalopta_FBEB':
    GRAPHENE_SEG = "graphene://https://local.cave.braininbrain.org/segmentation/table/megalopta_FB_EB_v3"
elif SPREADSHEET_NAME == 'Megalopta_PB':
    GRAPHENE_SEG = "graphene://https://local.cave.braininbrain.org/segmentation/table/megalopta_PB1_v2a"
elif SPREADSHEET_NAME == 'Megalopta_PB2':
    GRAPHENE_SEG = "graphene://https://local.cave.braininbrain.org/segmentation/table/megalopta_PB2"
elif SPREADSHEET_NAME == 'Eciton_NO':
    GRAPHENE_SEG = "graphene://https://local.cave.braininbrain.org/segmentation/table/eciton_NO_v5"
elif SPREADSHEET_NAME == 'Eciton_FBEB':
    GRAPHENE_SEG = "graphene://https://local.cave.braininbrain.org/segmentation/table/heinze_eciton_FB"
elif SPREADSHEET_NAME == 'Eciton_PB':
    GRAPHENE_SEG = "graphene://https://local.cave.braininbrain.org/segmentation/table/heinze_eciton_PB_v1"



# 4) Performance
BATCH_POINTS = 512     # number of coordinates to query per scattered_points() batch
SLEEP_BETWEEN_BATCHES = 0.0  # seconds; increase if you hit rate limits

# -----------------------
# INIT CLIENTS
# -----------------------
print("Initializing CAVE client...")
client = caveclient.CAVEclient(server_address=CAVE_SERVER)
client = caveclient.CAVEclient(datastack_name=DATASTACK_NAME)
cg = client.chunkedgraph
print("CAVE client ready.")

print("Initializing CloudVolume (graphene)...")
vol = cv.CloudVolume(GRAPHENE_SEG, use_https=True, progress=False)
print("CloudVolume ready.")

print("Initializing Google Sheets client...")

gc = gspread.oauth()
sh = gc.open_by_key(SPREADSHEET_ID)
print("Google Sheets connected.")


# -----------------------
# HELPERS
# -----------------------
def parse_xyz(s: str) -> Optional[Tuple[int, int, int]]:
    """Parse 'x, y, z' into (x, y, z) ints. Return None if invalid."""
    if not s:
        return None
    try:
        parts = [p.strip() for p in s.split(",")]
        if len(parts) != 3:
            return None
        return tuple(int(p) for p in parts)  # type: ignore
    except Exception:
        return None


def batched(iterable, n):
    """Yield lists of length n from iterable."""
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) >= n:
            yield batch
            batch = []
    if batch:
        yield batch


def list_worksheets(spreadsheet, target_names: List[str]) -> List[gspread.Worksheet]:
    """Return a list of worksheets filtered by target_names, excluding any with 'summary' in the title."""
    if target_names:
        name_set = set(target_names)
        return [
            ws for ws in spreadsheet.worksheets()
            if ws.title in name_set and "summary" not in ws.title.lower()
        ]
    else:
        return [
            ws for ws in spreadsheet.worksheets()
            if "summary" not in ws.title.lower()
        ]

def header_map(ws: gspread.Worksheet) -> Dict[str, int]:
    """Return dict: header -> 1-based column index; assumes headers are in first row."""
    header_row = ws.row_values(1)
    return {name: idx+1 for idx, name in enumerate(header_row)}


# def get_all_records_df(ws: gspread.Worksheet) -> pd.DataFrame:
#     """Read entire sheet into DataFrame with header row."""
#     data = ws.get_all_records()  # list of dicts
#     if not data:
#         # Build empty df with headers from row 1
#         headers = ws.row_values(1)
#         return pd.DataFrame(columns=headers)
#     return pd.DataFrame(data)

def get_all_records_df(ws: gspread.Worksheet) -> pd.DataFrame:
    data = ws.get_all_records(value_render_option='FORMATTED_VALUE')  # return strings
    if not data:
        headers = ws.row_values(1)
        return pd.DataFrame(columns=headers)
    return pd.DataFrame(data, dtype=object)  # keep as object, no coercion


def write_column_updates(ws: gspread.Worksheet,
                         df: pd.DataFrame,
                         col_name: str,
                         values: List[Optional[Union[str, int]]],
                         start_row: int = 2):
    """
    Write a list of values into a single named column, rows aligned with df.
    If the header doesn't exist, create it as a new column at the end.
    """
    header_row = ws.row_values(1)
    hmap = {name: idx+1 for idx, name in enumerate(header_row)}

    if col_name not in hmap:
        # Create the column header at the end
        new_col_idx = len(header_row) + 1
        ws.update_cell(1, new_col_idx, col_name)
        col_idx = new_col_idx
    else:
        col_idx = hmap[col_name]

    # Build a 2D list (one column) the size of df
    to_write = [[("" if v is None else str(v))] for v in values]
    end_row = start_row + len(to_write) - 1
    cell_range = gspread.utils.rowcol_to_a1(start_row, col_idx) + ":" + gspread.utils.rowcol_to_a1(end_row, col_idx)
    ws.update(cell_range, to_write, value_input_option="USER_ENTERED")

def ensure_column(ws: gspread.Worksheet, header_name: str) -> int:
    """
    Ensure a header exists in row 1. If missing, append it in the next available column.
    Returns the 1-based column index.
    """
    header = ws.row_values(1)

    # Existing column
    for i, h in enumerate(header, start=1):
        if h == header_name:
            return i

    # Append new column after current header row contents
    new_col_idx = len(header) + 1

    if new_col_idx > ws.col_count:
        ws.add_cols(new_col_idx - ws.col_count)

    ws.update(f"{gspread.utils.rowcol_to_a1(1, new_col_idx)}",
              [[header_name]],
              value_input_option="USER_ENTERED")
    return new_col_idx
    
def ensure_column_any(ws: gspread.Worksheet, preferred: str, aliases: List[str]) -> str:
    """
    Ensure one of (preferred + aliases) exists. If none exist, create the preferred one.
    Returns the column name that will be used downstream.
    """
    header = ws.row_values(1)
    # exact name if present
    if preferred in header:
        return preferred
    # otherwise, first alias that exists
    for a in aliases:
        if a in header:
            return a
    # none found: create preferred
    ensure_column(ws, preferred)
    return preferred

# -----------------------
# MAIN
# -----------------------
def update_supervoxels_for_sheet(ws: gspread.Worksheet):
    print(f"\nProcessing sheet: {ws.title}")

    # Ensure columns exist
    ensure_column(ws, COL_COORD)
    super_col_name = ensure_column_any(ws, COL_SUPER, aliases=["Supervoxel Id"])
    _ = ensure_column_any(ws, COL_ROOT, aliases=["Root Id"])  # keep naming consistent

    # Refresh header map after potential inserts
    hmap = header_map(ws)
    coord_idx = hmap[COL_COORD]
    super_idx = hmap[super_col_name]

    header, rows, last_row = get_sheet_matrix(ws)
    if last_row <= 1:
        print("  Sheet empty, skipping.")
        return

    points = []
    row2point = {}   # {sheet_row: (x,y,z) or None for bad coords}

    for sheet_row, row in enumerate(rows, start=2):
        coord_s = get_cell_str(row, coord_idx)
        if not coord_s:
            continue

        tup = parse_xyz(coord_s)
        if tup is None:
            row2point[sheet_row] = None
            continue

        row2point[sheet_row] = tup
        points.append(tup)

    if not row2point:
        print("  No coordinates found.")
        return

    print(f"  Querying supervoxels for {len(points)} points (batched @ {BATCH_POINTS})...")
    sv_results = {}

    for batch in tqdm(list(batched(points, BATCH_POINTS)), desc="  SV batches"):
        try:
            res = vol.scattered_points(batch)
            if res:
                sv_results.update(res)
        except Exception as e:
            print(f"    Warning: scattered_points batch failed with: {e}")
        if SLEEP_BETWEEN_BATCHES > 0:
            time.sleep(SLEEP_BETWEEN_BATCHES)

    values_by_row = {}
    for sheet_row, p in row2point.items():
        if p is None:
            values_by_row[sheet_row] = "Na"
        else:
            sv_id = sv_results.get(p)
            values_by_row[sheet_row] = str(sv_id) if sv_id is not None else "Na"

    print("  Writing Supervoxel IDs (row-safe, overwriting existing)...")
    write_full_column(ws, super_idx, values_by_row, last_row, start_row=2)

def get_sheet_matrix(ws: gspread.Worksheet):
    """
    Returns (header, rows, last_row).
    - header: row 1 list
    - rows: list of padded row lists for rows 2..last_row (preserves empty rows within range)
    - last_row: last row included from get_all_values()
    """
    all_vals = ws.get_all_values()
    if not all_vals:
        return [], [], 1

    header = all_vals[0]
    ncols = len(header)

    rows = []
    for r in all_vals[1:]:
        if len(r) < ncols:
            r = r + [""] * (ncols - len(r))
        rows.append(r)

    last_row = len(all_vals)
    return header, rows, last_row


def get_cell_str(row: list, col_idx_1based: int) -> str:
    j = col_idx_1based - 1
    if j < 0 or j >= len(row):
        return ""
    return (row[j] or "").strip()


def write_full_column(ws: gspread.Worksheet, col_idx: int, values_by_row: dict, last_row: int, start_row: int = 2):
    """
    Writes rows start_row..last_row into col_idx.
    values_by_row maps {sheet_row -> value}. Missing rows are written as "".
    """
    if last_row < start_row:
        return

    col_values = []
    for r in range(start_row, last_row + 1):
        v = values_by_row.get(r, "")
        col_values.append(["" if v is None else str(v)])

    a1 = gspread.utils.rowcol_to_a1(start_row, col_idx)
    b1 = gspread.utils.rowcol_to_a1(last_row, col_idx)
    ws.update(f"{a1}:{b1}", col_values, value_input_option="USER_ENTERED")


def update_roots_for_sheet(ws: gspread.Worksheet):
    print(f"\nUpdating roots for sheet: {ws.title}")

    super_col_name = ensure_column_any(ws, COL_SUPER, aliases=["Supervoxel Id"])
    root_col_name  = ensure_column_any(ws, COL_ROOT,  aliases=["Root Id"])
    ensure_column(ws, COL_ROOT_UPDATED)

    hmap = header_map(ws)
    super_idx = hmap[super_col_name]
    root_idx = hmap[root_col_name]
    updated_idx = hmap[COL_ROOT_UPDATED]

    header, rows, last_row = get_sheet_matrix(ws)
    if last_row <= 1:
        print("  Sheet empty, skipping.")
        return

    # Collect SVs by sheet row
    sv_by_row = {}
    for sheet_row, row in enumerate(rows, start=2):
        sv_s = get_cell_str(row, super_idx)
        if not sv_s or sv_s.lower() == "na":
            continue
        try:
            sv_by_row[sheet_row] = int(sv_s)
        except Exception:
            # malformed value -> skip
            continue

    if not sv_by_row:
        print("  No valid supervoxels found to update roots.")
        return

    sheet_rows = list(sv_by_row.keys())
    sv_ids = [sv_by_row[r] for r in sheet_rows]

    print(f"  Querying roots for {len(sv_ids)} supervoxels...")
    roots_by_row = {}

    try:
        roots = cg.get_roots(sv_ids)
        for r, root_val in zip(sheet_rows, roots):
            if root_val is None:
                roots_by_row[r] = "Na"
            elif isinstance(root_val, (list, tuple)) and len(root_val) > 0:
                roots_by_row[r] = str(root_val[0])
            else:
                roots_by_row[r] = str(root_val)
    except Exception as e:
        print(f"  Batch get_roots failed ({e}); falling back to per-SV.")
        for r in tqdm(sheet_rows, desc="  roots per-SV"):
            sv = sv_by_row[r]
            try:
                root_val = cg.get_roots(sv)
                if root_val is None:
                    roots_by_row[r] = "Na"
                elif isinstance(root_val, (list, tuple)) and len(root_val) > 0:
                    roots_by_row[r] = str(root_val[0])
                else:
                    roots_by_row[r] = str(root_val)
            except Exception:
                roots_by_row[r] = "Na"

    print("  Writing Root IDs (row-safe)...")
    write_full_column(ws, root_idx, roots_by_row, last_row, start_row=2)

    # Update the date column only where we wrote a non-Na root
    updated_by_row = {}
    for r, root_str in roots_by_row.items():
        if root_str and root_str.lower() != "na":
            updated_by_row[r] = RUN_DATE_STR

    print(f"  Writing '{COL_ROOT_UPDATED}' dates (row-safe)...")
    write_full_column(ws, updated_idx, updated_by_row, last_row, start_row=2)



def main():
    worksheets = list_worksheets(sh, SHEET_NAMES)

    # 1) Fill Supervoxel IDs from coordinates
    for ws in worksheets:
        update_supervoxels_for_sheet(ws)

    # 2) Fill Root IDs from Supervoxel IDs
    for ws in worksheets:
        update_roots_for_sheet(ws)

    print("\nAll done.")


# if __name__ == "__main__":
#     main()

Initializing CAVE client...
CAVE client ready.
Initializing CloudVolume (graphene)...
CloudVolume ready.
Initializing Google Sheets client...
Google Sheets connected.


In [22]:
# run script
main()


Processing sheet: megalopta_pb2_delta7
  Querying supervoxels for 41 points (batched @ 512)...


  SV batches: 100%|██████████████████████████████████████████████████████████████████████| 1/1 [00:04<00:00,  5.00s/it]
C:\Users\Marcel\AppData\Local\Temp\ipykernel_10932\3218394753.py:336: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  ws.update(f"{a1}:{b1}", col_values, value_input_option="USER_ENTERED")


  Writing Supervoxel IDs (row-safe, overwriting existing)...

Processing sheet: megalopta_pb2_EPG
  Querying supervoxels for 4 points (batched @ 512)...


  SV batches: 100%|██████████████████████████████████████████████████████████████████████| 1/1 [00:03<00:00,  3.50s/it]


  Writing Supervoxel IDs (row-safe, overwriting existing)...

Processing sheet: megalopta_pb2_PFN
  Querying supervoxels for 7 points (batched @ 512)...


  SV batches: 100%|██████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.41s/it]


  Writing Supervoxel IDs (row-safe, overwriting existing)...

Processing sheet: megalopta_pb2_PEN
  Querying supervoxels for 2 points (batched @ 512)...


  SV batches: 100%|██████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.05s/it]


  Writing Supervoxel IDs (row-safe, overwriting existing)...

Updating roots for sheet: megalopta_pb2_delta7
  Querying roots for 41 supervoxels...
  Writing Root IDs (row-safe)...
  Writing 'Root ID updated' dates (row-safe)...

Updating roots for sheet: megalopta_pb2_EPG
  Querying roots for 4 supervoxels...
  Writing Root IDs (row-safe)...
  Writing 'Root ID updated' dates (row-safe)...

Updating roots for sheet: megalopta_pb2_PFN
  Querying roots for 7 supervoxels...
  Writing Root IDs (row-safe)...
  Writing 'Root ID updated' dates (row-safe)...

Updating roots for sheet: megalopta_pb2_PEN
  Querying roots for 2 supervoxels...
  Writing Root IDs (row-safe)...
  Writing 'Root ID updated' dates (row-safe)...

All done.
